# Aarogya — Extensive Fine-tuning of Gemma 4 4B (Unsloth + QLoRA)

**Hackathon:** Gemma 4 Good · Special Track: **Unsloth** ($10K)

**Hardware:** Kaggle GPU **T4 × 2** · Internet **ON**

**Runtime:** ~3-4 hours for 1500 steps on 5000 samples

Produces a LoRA adapter that makes Gemma 4 better at:
- Hindi/English code-mixed rural symptom descriptions
- Strict JSON triage output (urgency, red flags, ASHA action note)
- Rural-context red-flag recognition (paani-wale dast, bachcha sust, etc.)

## 1. Install dependencies

In [ ]:
!pip install -qU unsloth==2024.10.7 trl==0.11.4 transformers==4.45.2 datasets==3.0.1 bitsandbytes==0.44.1 accelerate==1.0.1 peft==0.13.0
!pip install -qU "xformers<0.0.27" || true

## 2. Clone the Aarogya repo

In [ ]:
import os
if not os.path.exists('/kaggle/working/aarogya'):
    !git clone https://github.com/shabdkumar3/aarogya.git /kaggle/working/aarogya
%cd /kaggle/working/aarogya/finetuning

## 3. Build the dataset (~5000 train / 800 val)

In [ ]:
!python prepare_dataset.py
!ls -la train_data.jsonl val_data.jsonl

## 4. Load Gemma 4 4B with QLoRA (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-4-4b-it',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Larger LoRA for more thorough adaptation (r=32 vs default r=16)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Trainable params:')
model.print_trainable_parameters()

## 5. Train (1500 steps · ~3-4h on T4×2)

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_ds = load_dataset('json', data_files='train_data.jsonl', split='train')
val_ds   = load_dataset('json', data_files='val_data.jsonl',   split='train')
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        max_steps=1500,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy='steps',
        eval_steps=150,
        save_strategy='steps',
        save_steps=300,
        save_total_limit=3,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/kaggle/working/aarogya-gemma4-checkpoints',
        report_to='none',
    ),
)
trainer_stats = trainer.train()
print(trainer_stats)

## 6. Save LoRA adapter (small, ~100 MB)

In [ ]:
model.save_pretrained('/kaggle/working/aarogya-gemma4-lora')
tokenizer.save_pretrained('/kaggle/working/aarogya-gemma4-lora')
!ls -lh /kaggle/working/aarogya-gemma4-lora

## 7. Export merged GGUF for Ollama edge inference

In [ ]:
model.save_pretrained_gguf(
    '/kaggle/working/aarogya-gemma4-gguf',
    tokenizer,
    quantization_method='q4_k_m',
)
!ls -lh /kaggle/working/aarogya-gemma4-gguf

## 8. Ablation: base Gemma 4 vs fine-tuned (for the Kaggle writeup)

In [ ]:
%cd /kaggle/working/aarogya
!python finetuning/evaluate.py --base unsloth/gemma-4-4b-it --finetuned /kaggle/working/aarogya-gemma4-lora || echo 'evaluate.py may need editing for your needs - see the script'

## 9. (Optional) Push the LoRA adapter to HuggingFace Hub

If you want judges to be able to download/use the model, push to HF Hub. Skip if just submitting Kaggle notebook URL.

In [ ]:
# from huggingface_hub import login
# login(token='hf_YOUR_TOKEN')
# model.push_to_hub('shabdkumar3/aarogya-gemma4-4b-lora')
# tokenizer.push_to_hub('shabdkumar3/aarogya-gemma4-4b-lora')